# Build Temporal Lagged Dataset

Строит третий датасет для прогноза минутной лог-доходности:

`y(t) = log(close(t+1) / close(t))`

Источник: исправленный общий датасет `model_artifacts/adausdt_minute_log_return_dataset.parquet`.

Принцип против leakage:

- `log_return_1m` и `return_1m` используются только как лаги/rolling по прошлым значениям через `shift(1)`;
- текущие микроструктурные признаки минуты `t` используются как известные после закрытия минуты `t`;
- rolling для текущих микроструктурных признаков включает текущую минуту `t`, потому что цель относится к `t+1`.

In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 140)

try:
    display
except NameError:
    display = print

## Settings

In [11]:
INPUT_DATASET = Path("model_artifacts/adausdt_minute_log_return_dataset.parquet")
OUTPUT_DATASET = Path("model_artifacts/adausdt_minute_log_return_temporal_dataset.parquet")
REPORT_PATH = Path("model_artifacts/temporal_dataset_report.json")

TARGET_COL = "log_return_1m"
RAW_RETURN_COL = "return_1m"
TIME_COLS = ["date", "timestamp"]

# Если True, оставляем базовые текущие микроструктурные признаки, которые входят в список ниже.
# Target/current return columns всё равно не попадут как текущие фичи.
KEEP_SELECTED_CURRENT_FEATURES = True

# Первые строки будут удалены из-за лагов/rolling. Максимальное окно здесь 60.
MAX_EXPECTED_WARMUP_ROWS = 60

## Load Base Dataset

In [12]:
df = pd.read_parquet(INPUT_DATASET)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)
display(df[["date", "timestamp", TARGET_COL, RAW_RETURN_COL]].head())
display(df[["date", "timestamp", TARGET_COL, RAW_RETURN_COL]].tail())

(3157919, 93)


,date,timestamp,log_return_1m,return_1m
0,2020-02-01,2020-02-01 00:01:00+00:00,-0.000744,-0.000743
1,2020-02-01,2020-02-01 00:02:00+00:00,0.000186,0.000186
2,2020-02-01,2020-02-01 00:03:00+00:00,-0.001861,-0.001859
3,2020-02-01,2020-02-01 00:04:00+00:00,-0.000745,-0.000745
4,2020-02-01,2020-02-01 00:05:00+00:00,0.000932,0.000932


,date,timestamp,log_return_1m,return_1m
3157914,2026-02-01,2026-02-01 23:55:00+00:00,0.000700,0.000701
3157915,2026-02-01,2026-02-01 23:56:00+00:00,0.001399,0.001400
3157916,2026-02-01,2026-02-01 23:57:00+00:00,-0.002099,-0.002097
3157917,2026-02-01,2026-02-01 23:58:00+00:00,0.000350,0.000350
3157918,2026-02-01,2026-02-01 23:59:00+00:00,0.001050,0.001050


## Feature Spec

In [13]:
RETURN_LAGS = [1, 2, 3, 5, 8, 13, 21]

AGGRESSION_COLS = [
    "aggression_features__delta_base_norm",
    "aggression_features__delta_quote_norm",
    "aggression_features__V_buy_quote",
    "aggression_features__V_sell_quote",
]
AGGRESSION_LAGS = [1, 2, 3, 5, 10, 15, 30]
AGGRESSION_WINDOWS = [5, 15, 30, 60]

TRADES_MINUTE_COLS = [
    "trades_minute_level__delta_quote_norm",
    "trades_minute_level__delta_base_norm",
]
TRADES_MINUTE_WINDOWS = [5, 15, 30]

PRICE_PRESSURE_COLS = [
    "price_pressure__PI_total",
    "price_pressure__F_PI",
    "price_pressure__F_eff",
    "price_pressure__F_asymmetry",
    "price_pressure__VWAP_pos",
]
PRICE_PRESSURE_LAGS = [1, 2, 3, 5, 10]
PRICE_PRESSURE_WINDOWS = [5, 15, 30]

INTRAMINUTE_DYNAMICS_COLS = [
    "intraminute_dynamics__pressure_total",
    "intraminute_dynamics__pressure_segment_3",
    "intraminute_dynamics__F_concentration",
    "intraminute_dynamics__RV",
]
INTRAMINUTE_DYNAMICS_WINDOWS = [5, 15, 30]

VWAP_SEGMENT_COLS = [
    "intraminute_segments__VWAP_minute",
    "intraminute_segments__VWAP_segment_1",
    "intraminute_segments__VWAP_segment_2",
    "intraminute_segments__VWAP_segment_3",
]
VWAP_SPREAD_COLS = [
    "vwap_spread__segment_3_minus_1",
    "vwap_spread__segment_3_minus_minute",
    "vwap_spread__segment_2_minus_1",
]
VWAP_SPREAD_WINDOWS = [5, 15, 30]

CANDLE_COLS = [
    "intraminute_features__body_norm",
    "intraminute_features__wick_upper_norm",
    "intraminute_features__wick_lower_norm",
    "intraminute_features__taker_buy_ratio",
]
CANDLE_LAGS = [1, 2, 3, 5]
CANDLE_WINDOWS = [5, 15, 30]

BTC_COLS = [
    "btc_features__btc_log_return",
    "btc_features__btc_zscore",
    "btc_features__btc_rolling_volatility",
]
BTC_LAGS = [1, 2, 3, 5, 10, 15, 30, 60]
BTC_WINDOWS = [5, 15, 30, 60]

TRADE_DISTRIBUTION_COLS = [
    "trade_distribution__N_eff_buy",
    "trade_distribution__N_eff_sell",
    "trade_distribution__C_eff_buy",
    "trade_distribution__C_eff_sell",
]
TRADE_DISTRIBUTION_WINDOWS = [15, 30, 60]

TIME_FEATURE_COLS = [
    "sinthetic_data__cos_hour",
    "sinthetic_data__sin_hour",
    "sinthetic_data__cos_weekday",
    "sinthetic_data__sin_weekday",
    "sinthetic_data__asia_open",
    "sinthetic_data__europe_open",
    "sinthetic_data__us_open",
]

In [14]:
required_cols = sorted(
    set([TARGET_COL, RAW_RETURN_COL, "timestamp", "date"])
    | set(AGGRESSION_COLS)
    | set(TRADES_MINUTE_COLS)
    | set(PRICE_PRESSURE_COLS)
    | set(INTRAMINUTE_DYNAMICS_COLS)
    | set(VWAP_SEGMENT_COLS)
    | set(CANDLE_COLS)
    | set(BTC_COLS)
    | set(TRADE_DISTRIBUTION_COLS)
    | set(TIME_FEATURE_COLS)
)
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

print(f"required columns ok: {len(required_cols)}")

required columns ok: 41


## Helpers

In [15]:
created_feature_groups = []


def add_lags(out, source, lags, group_name):
    created = []
    for col in source:
        for lag in lags:
            name = f"{col}__lag_{lag}"
            out[name] = df[col].shift(lag)
            created.append(name)
    created_feature_groups.append({"group": group_name, "kind": "lags", "count": len(created)})
    return created


def get_feature_series(out, col):
    if col in out.columns:
        return out[col]
    return df[col]


def add_rolling(out, source, windows, funcs, group_name, shift_before=False):
    created = []
    for col in source:
        base_series = get_feature_series(out, col)
        series = base_series.shift(1) if shift_before else base_series
        for window in windows:
            rolling = series.rolling(window=window, min_periods=window)
            if "mean" in funcs:
                name = f"{col}__roll_mean_{window}"
                out[name] = rolling.mean()
                created.append(name)
            if "std" in funcs:
                name = f"{col}__roll_std_{window}"
                out[name] = rolling.std(ddof=0)
                created.append(name)
            if "sum" in funcs:
                name = f"{col}__roll_sum_{window}"
                out[name] = rolling.sum()
                created.append(name)
            if "abs_sum" in funcs:
                name = f"{col}__roll_abs_sum_{window}"
                out[name] = series.abs().rolling(window=window, min_periods=window).sum()
                created.append(name)
            if "max" in funcs:
                name = f"{col}__roll_max_{window}"
                out[name] = rolling.max()
                created.append(name)
    created_feature_groups.append({"group": group_name, "kind": "rolling", "count": len(created)})
    return created


def add_ewma(out, source, spans, group_name, shift_before=False):
    created = []
    for col in source:
        base_series = get_feature_series(out, col)
        series = base_series.shift(1) if shift_before else base_series
        for span in spans:
            name = f"{col}__ewm_mean_{span}"
            out[name] = series.ewm(span=span, adjust=False, min_periods=span).mean()
            created.append(name)
    created_feature_groups.append({"group": group_name, "kind": "ewma", "count": len(created)})
    return created

## Build Temporal Features

In [16]:
out = df[["date", "timestamp", TARGET_COL, RAW_RETURN_COL]].copy()

selected_current_cols = sorted(
    set(AGGRESSION_COLS)
    | set(TRADES_MINUTE_COLS)
    | set(PRICE_PRESSURE_COLS)
    | set(INTRAMINUTE_DYNAMICS_COLS)
    | set(VWAP_SEGMENT_COLS)
    | set(CANDLE_COLS)
    | set(BTC_COLS)
    | set(TRADE_DISTRIBUTION_COLS)
    | set(TIME_FEATURE_COLS)
)

if KEEP_SELECTED_CURRENT_FEATURES:
    out = pd.concat([out, df[selected_current_cols]], axis=1)
    created_feature_groups.append({"group": "selected_current_features", "kind": "current", "count": len(selected_current_cols)})

# 1. Price returns, lagged only to avoid target leakage.
add_lags(out, [TARGET_COL, RAW_RETURN_COL], RETURN_LAGS, "returns")

# 2. Aggression flow.
add_lags(out, AGGRESSION_COLS, AGGRESSION_LAGS, "aggression_flow")
add_rolling(out, AGGRESSION_COLS[:2], AGGRESSION_WINDOWS, ["sum", "mean", "std", "abs_sum"], "aggression_delta")
add_ewma(out, AGGRESSION_COLS[:2], AGGRESSION_WINDOWS, "aggression_delta")

# 3. Trades minute level, rolling only.
add_rolling(out, TRADES_MINUTE_COLS, TRADES_MINUTE_WINDOWS, ["mean", "std", "sum"], "trades_minute_level")

# 4. Price pressure.
add_lags(out, PRICE_PRESSURE_COLS, PRICE_PRESSURE_LAGS, "price_pressure")
add_rolling(out, PRICE_PRESSURE_COLS, PRICE_PRESSURE_WINDOWS, ["mean", "std"], "price_pressure")
add_ewma(out, PRICE_PRESSURE_COLS, PRICE_PRESSURE_WINDOWS, "price_pressure")

# 5. Intraminute dynamics, rolling only.
add_rolling(out, INTRAMINUTE_DYNAMICS_COLS, INTRAMINUTE_DYNAMICS_WINDOWS, ["mean", "std", "max"], "intraminute_dynamics")

# 6. VWAP segment spreads and rolling.
out["vwap_spread__segment_3_minus_1"] = df["intraminute_segments__VWAP_segment_3"] - df["intraminute_segments__VWAP_segment_1"]
out["vwap_spread__segment_3_minus_minute"] = df["intraminute_segments__VWAP_segment_3"] - df["intraminute_segments__VWAP_minute"]
out["vwap_spread__segment_2_minus_1"] = df["intraminute_segments__VWAP_segment_2"] - df["intraminute_segments__VWAP_segment_1"]
created_feature_groups.append({"group": "vwap_spreads", "kind": "spreads", "count": len(VWAP_SPREAD_COLS)})
add_rolling(out, VWAP_SPREAD_COLS, VWAP_SPREAD_WINDOWS, ["mean", "std"], "vwap_spreads")

# 7. Candle features.
add_lags(out, CANDLE_COLS, CANDLE_LAGS, "candle_features")
add_rolling(out, CANDLE_COLS, CANDLE_WINDOWS, ["mean", "std"], "candle_features")

# 8. BTC features.
add_lags(out, BTC_COLS, BTC_LAGS, "btc_features")
add_rolling(out, BTC_COLS, BTC_WINDOWS, ["mean", "std"], "btc_features")
add_ewma(out, BTC_COLS, BTC_WINDOWS, "btc_features")

# 9. Trade distribution, rolling only.
add_rolling(out, TRADE_DISTRIBUTION_COLS, TRADE_DISTRIBUTION_WINDOWS, ["mean", "std", "sum"], "trade_distribution")

print(out.shape)
display(pd.DataFrame(created_feature_groups))

C:\Users\Пользователь\AppData\Local\Temp\ipykernel_38132\4265839001.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = rolling.std(ddof=0)
C:\Users\Пользователь\AppData\Local\Temp\ipykernel_38132\4265839001.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = rolling.sum()
C:\Users\Пользователь\AppData\Local\Temp\ipykernel_38132\4265839001.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joi

(3157919, 404)


C:\Users\Пользователь\AppData\Local\Temp\ipykernel_38132\4265839001.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = rolling.std(ddof=0)
C:\Users\Пользователь\AppData\Local\Temp\ipykernel_38132\4265839001.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = rolling.sum()


,group,kind,count
0,selected_current_features,current,37
1,returns,lags,14
2,aggression_flow,lags,28
3,aggression_delta,rolling,32
4,aggression_delta,ewma,8
5,trades_minute_level,rolling,18
6,price_pressure,lags,25
7,price_pressure,rolling,30
8,price_pressure,ewma,15
9,intraminute_dynamics,rolling,36


## Cleanup And Validation

In [17]:
feature_cols = [col for col in out.columns if col not in {"date", "timestamp", TARGET_COL, RAW_RETURN_COL}]

missing_before = out[feature_cols + [TARGET_COL]].isna().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)
rows_before = len(out)

clean = out.dropna(subset=feature_cols + [TARGET_COL]).reset_index(drop=True)
rows_after = len(clean)
dropped_rows = rows_before - rows_after

constant_feature_cols = [col for col in feature_cols if clean[col].nunique(dropna=False) <= 1]
if constant_feature_cols:
    print("Dropping constant feature columns:")
    print(constant_feature_cols)
    clean = clean.drop(columns=constant_feature_cols)
    feature_cols = [col for col in feature_cols if col not in constant_feature_cols]

missing_after = int(clean[feature_cols + [TARGET_COL]].isna().sum().sum())

print("rows_before", rows_before)
print("rows_after", rows_after)
print("dropped_rows", dropped_rows)
print("missing_after", missing_after)
print("feature_cols", len(feature_cols))
display(clean[["date", "timestamp", TARGET_COL, RAW_RETURN_COL]].head())
display(clean[["date", "timestamp", TARGET_COL, RAW_RETURN_COL]].tail())

if dropped_rows > MAX_EXPECTED_WARMUP_ROWS + 5:
    raise RuntimeError(f"Dropped too many rows: {dropped_rows}")
if missing_after:
    raise RuntimeError(f"Missing values remain after cleanup: {missing_after}")

rows_before 3157919
rows_after 3157859
dropped_rows 60
missing_after 0
feature_cols 400


,date,timestamp,log_return_1m,return_1m
0,2020-02-01,2020-02-01 01:01:00+00:00,0.000000,0.000000
1,2020-02-01,2020-02-01 01:02:00+00:00,0.000919,0.000920
2,2020-02-01,2020-02-01 01:03:00+00:00,-0.001287,-0.001286
3,2020-02-01,2020-02-01 01:04:00+00:00,0.001839,0.001840
4,2020-02-01,2020-02-01 01:05:00+00:00,-0.000919,-0.000918


,date,timestamp,log_return_1m,return_1m
3157854,2026-02-01,2026-02-01 23:55:00+00:00,0.000700,0.000701
3157855,2026-02-01,2026-02-01 23:56:00+00:00,0.001399,0.001400
3157856,2026-02-01,2026-02-01 23:57:00+00:00,-0.002099,-0.002097
3157857,2026-02-01,2026-02-01 23:58:00+00:00,0.000350,0.000350
3157858,2026-02-01,2026-02-01 23:59:00+00:00,0.001050,0.001050


## Save Dataset And Report

In [18]:
OUTPUT_DATASET.parent.mkdir(parents=True, exist_ok=True)
clean.to_parquet(OUTPUT_DATASET, index=False, compression="zstd")

report = {
    "input_dataset": str(INPUT_DATASET),
    "output_dataset": str(OUTPUT_DATASET),
    "rows_before": int(rows_before),
    "rows_after": int(rows_after),
    "dropped_rows": int(dropped_rows),
    "columns": int(clean.shape[1]),
    "feature_columns": int(len(feature_cols)),
    "target": TARGET_COL,
    "raw_return_column_kept_for_diagnostics": RAW_RETURN_COL,
    "missing_before_top": {col: int(value) for col, value in missing_before.head(50).items()},
    "missing_after": int(missing_after),
    "constant_feature_columns_dropped": constant_feature_cols,
    "feature_groups": created_feature_groups,
    "time_range": {
        "start": str(clean["timestamp"].min()),
        "end": str(clean["timestamp"].max()),
    },
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(f"saved dataset: {OUTPUT_DATASET}")
print(f"saved report: {REPORT_PATH}")
print(report)

saved dataset: model_artifacts\adausdt_minute_log_return_temporal_dataset.parquet
saved report: model_artifacts\temporal_dataset_report.json
{'input_dataset': 'model_artifacts\\adausdt_minute_log_return_dataset.parquet', 'output_dataset': 'model_artifacts\\adausdt_minute_log_return_temporal_dataset.parquet', 'rows_before': 3157919, 'rows_after': 3157859, 'dropped_rows': 60, 'columns': 404, 'feature_columns': 400, 'target': 'log_return_1m', 'raw_return_column_kept_for_diagnostics': 'return_1m', 'missing_before_top': {'btc_features__btc_zscore__lag_60': 60, 'btc_features__btc_rolling_volatility__lag_60': 60, 'btc_features__btc_log_return__lag_60': 60, 'trade_distribution__C_eff_sell__roll_mean_60': 59, 'trade_distribution__C_eff_sell__roll_sum_60': 59, 'trade_distribution__C_eff_sell__roll_std_60': 59, 'btc_features__btc_rolling_volatility__roll_std_60': 59, 'aggression_features__delta_base_norm__roll_abs_sum_60': 59, 'aggression_features__delta_base_norm__roll_sum_60': 59, 'btc_features